In [1]:
# Import libraries
import os
import csv
import gzip
import pickle
import numpy as np
import pandas as pd
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
from pxblat import Server, Client #pxblat==0.3.6

/Users/vivianschu/opt/anaconda3/envs/scgpt/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define model (finetuned or pretrained)
model = 'random_init-2'
dataset = 'pancreas'

In [3]:
# Define the path to the directory containing the pickle files
base = "/Users/vivianschu/Documents/MBP/Lab/Genomic Interpretability/scgpt_attn_results/test/FINAL_LRP-attention"
data_dir = os.path.join(base, dataset, model)

pkl_files = [
    'examples_scores_attention_layer0.p',
    'examples_scores_attention_layer1.p',
    'examples_scores_attention_layer2.p',
    'examples_scores_attention_layer3.p',
    'examples_scores_attention_layer4.p',
    'examples_scores_attention_layer5.p',
    'examples_scores_attention_layer6.p',
    'examples_scores_attention_layer7.p',
    'examples_scores_attention_layer8.p',
    'examples_scores_attention_layer9.p',
    'examples_scores_attention_layer10.p',
    'examples_scores_attention_layer11.p',
]


In [4]:
# Load all pickle files into memory
layers_data = []
for file_name in pkl_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'rb') as f:
        layers_data.append(pickle.load(f))

# Assuming all layers have the same number of heads and all heads have the same number of cells
num_layers = len(layers_data)
num_heads = len(layers_data[0])
num_cells = len(layers_data[0][0])
num_genes = len(layers_data[0][0][0][0]) - 1  # Assuming each cell contains data for the same number of genes

print('Check Data:', num_layers, num_heads, num_cells, num_genes)


Check Data: 12 8 499 499


In [5]:
# Open compiled_cells.csv
compiled_cells_path = os.path.join(data_dir, 'compiled_cells.csv')
compiled_cells = pd.read_csv(compiled_cells_path, sep=';')

compiled_cells.head()

,gene_sequence,label,expression,layer0_head0,layer0_head1,layer0_head2,layer0_head3,layer0_head4,layer0_head5,layer0_head6,...,layer11_head2,layer11_head3,layer11_head4,layer11_head5,layer11_head6,layer11_head7,position_first_third,position_middle_third,position_last_third,position
0,"<cls>,TXLNA,ATP2A3,ZNF141,CCDC141,SLC25A37,STO...",5,"0.0,48.0,33.0,25.0,5.0,3.0,10.0,28.0,5.0,29.0,...","0.0023745133,0.0032477628,0.003589545,0.003814...","0.0041318787,0.0053796424,0.0041540335,0.00306...","0.0056217974,0.0031721308,0.005039241,0.004776...","0.0034448185,0.0032406966,0.0040769065,0.00487...","0.0038653233,0.0041617695,0.0020131315,0.00385...","0.003914756,0.0034802882,0.005446018,0.0034722...","0.0042973505,0.0035008143,0.0028890239,0.00347...",...,"0.0029499263,0.0024055955,0.002513074,0.002640...","0.0025008207,0.0026199492,0.002841934,0.002556...","0.00444397,0.0023903793,0.0024260334,0.0031343...","0.002858873,0.0026262312,0.0025715895,0.002385...","0.0030605097,0.002257548,0.0024005037,0.002462...","0.0050789197,0.0029359038,0.003200909,0.003342...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,1..."
1,"<cls>,DSG2,TMEM144,TNFRSF1B,FKBP11,MUC20,DCN,A...",5,"0.0,39.0,7.0,19.0,6.0,17.0,7.0,19.0,26.0,6.0,4...","0.0022430576,0.0024487586,0.003219843,0.003661...","0.004013164,0.0031627351,0.0036624381,0.003953...","0.0042283423,0.0031852438,0.004393687,0.002575...","0.0034300121,0.0023620245,0.0043808324,0.00346...","0.0032802566,0.0028360444,0.0044557275,0.00289...","0.004048269,0.0047665904,0.0038825262,0.003081...","0.0042852582,0.0031577605,0.0056061805,0.00359...",...,"0.0029518714,0.002717198,0.0027214303,0.002277...","0.0027387682,0.0030519136,0.0027144155,0.00291...","0.0033776136,0.002504747,0.0026376792,0.002287...","0.0028611864,0.0027302948,0.002847653,0.002169...","0.0032267557,0.0026814435,0.0026326487,0.00242...","0.004437197,0.0030190656,0.0034014382,0.002829...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,1..."
2,"<cls>,CNN2,LRAT,ACTN1,PLD6,PHLDB2,PIGA,CES2,MY...",5,"0.0,27.0,20.0,39.0,5.0,7.0,3.0,11.0,12.0,1.0,3...","0.0022238698,0.0040896675,0.004484682,0.002118...","0.0039243894,0.0040566986,0.0026453421,0.00478...","0.0048712324,0.0034570356,0.0030926508,0.00369...","0.003476348,0.0037290629,0.0028929042,0.004210...","0.003399026,0.0025909895,0.0042975056,0.004100...","0.004021946,0.002352441,0.0031674122,0.0027053...","0.0031837898,0.0027910753,0.0021618963,0.00399...",...,"0.0027702104,0.002785568,0.002417999,0.0025976...","0.0025693388,0.0023259071,0.002565366,0.002743...","0.0035079145,0.0025205663,0.002532903,0.002934...","0.0025086715,0.0025067604,0.0024263477,0.00244...","0.002859213,0.002560808,0.0023383668,0.0023868...","0.004376093,0.0029561946,0.0025371308,0.003263...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,1..."
3,"<cls>,NINJ1,TNS3,SLC6A6,PGM2L1,CABYR,SLC3A1,SE...",5,"0.0,42.0,19.0,39.0,20.0,17.0,14.0,33.0,21.0,37...","0.0020445194,0.0031070835,0.0025515305,0.00327...","0.0040934836,0.0028073455,0.0034256505,0.00244...","0.0043800664,0.0029222732,0.0025062195,0.00437...","0.0034533436,0.0025593678,0.003978598,0.002954...","0.0038189043,0.0032943436,0.002777985,0.003936...","0.0036355853,0.0033676857,0.00409,0.0038118216...","0.0043353783,0.0040053073,0.002787665,0.002944...",...,"0.0028078433,0.002296238,0.0024589011,0.002672...","0.002720637,0.002522182,0.0024271347,0.0025703...","0.0036673287,0.0026340187,0.002360998,0.002149...","0.0025690747,0.0023017398,0.0025352214,0.00236...","0.0029805924,0.002354244,0.0025769223,0.0026

In [6]:
# Open compiled_cells_features.csv
cells_features_path = os.path.join(data_dir, 'compiled_cells-features.csv')
cells_features = pd.read_csv(cells_features_path, sep=';')

cells_features.head()

,gene_sequence,label,expression,layer0_head0,layer0_head1,layer0_head2,layer0_head3,layer0_head4,layer0_head5,layer0_head6,...,HP_X_LINKED_DOMINANT_INHERITANCE,HP_X_LINKED_RECESSIVE_INHERITANCE,HP_YELLOW_BROWN_DISCOLORATION_OF_THE_TEETH,HP_YELLOW_WHITE_LESIONS_OF_THE_MACULA,HP_YELLOW_WHITE_LESIONS_OF_THE_RETINA,HP_YOUNG_ADULT_ONSET,HP_Y_LINKED_INHERITANCE,HP_Y_SHAPED_METACARPALS,HP_ZOLLINGER_ELLISON_SYNDROME,HP_ZONULAR_CATARACT
0,"<cls>,TXLNA,ATP2A3,ZNF141,CCDC141,SLC25A37,STO...",5,"0.0,48.0,33.0,25.0,5.0,3.0,10.0,28.0,5.0,29.0,...","0.0023745133,0.0032477628,0.003589545,0.003814...","0.0041318787,0.0053796424,0.0041540335,0.00306...","0.0056217974,0.0031721308,0.005039241,0.004776...","0.0034448185,0.0032406966,0.0040769065,0.00487...","0.0038653233,0.0041617695,0.0020131315,0.00385...","0.003914756,0.0034802882,0.005446018,0.0034722...","0.0042973505,0.0035008143,0.0028890239,0.00347...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
1,"<cls>,DSG2,TMEM144,TNFRSF1B,FKBP11,MUC20,DCN,A...",5,"0.0,39.0,7.0,19.0,6.0,17.0,7.0,19.0,26.0,6.0,4...","0.0022430576,0.0024487586,0.003219843,0.003661...","0.004013164,0.0031627351,0.0036624381,0.003953...","0.0042283423,0.0031852438,0.004393687,0.002575...","0.0034300121,0.0023620245,0.0043808324,0.00346...","0.0032802566,0.0028360444,0.0044557275,0.00289...","0.004048269,0.0047665904,0.0038825262,0.003081...","0.0042852582,0.0031577605,0.0056061805,0.00359...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
2,"<cls>,CNN2,LRAT,ACTN1,PLD6,PHLDB2,PIGA,CES2,MY...",5,"0.0,27.0,20.0,39.0,5.0,7.0,3.0,11.0,12.0,1.0,3...","0.0022238698,0.0040896675,0.004484682,0.002118...","0.0039243894,0.0040566986,0.0026453421,0.00478...","0.0048712324,0.0034570356,0.0030926508,0.00369...","0.003476348,0.0037290629,0.0028929042,0.004210...","0.003399026,0.0025909895,0.0042975056,0.004100...","0.004021946,0.002352441,0.0031674122,0.0027053...","0.0031837898,0.0027910753,0.0021618963,0.00399...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
3,"<cls>,NINJ1,TNS3,SLC6A6,PGM2L1,CABYR,SLC3A1,SE...",5,"0.0,42.0,19.0,39.0,20.0,17.0,14.0,33.0,21.0,37...","0.0020445194,0.0031070835,0.0025515305,0.00327...","0.0040934836,0.0028073455,0.0034256505,0.00244...","0.0043800664,0.0029222732,0.0025062195,0.00437...","0.0034533436,0.0025593678,0.003978598,0.002954...","0.0038189043,0.0032943436,0.002777985,0.003936...","0.0036355853,0.0033676857,0.00409,0.0038118216...","0.0043353783,0.0040053073,0.002787665,0.002944...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",

In [7]:
cells_features.shape

(499, 16326)

In [8]:
# Get column names
column_names = cells_features.columns[103:].tolist()
print(f"Number of columns: {len(column_names)}")

# Print column names line by line
print("Column names:")
for col in column_names:
    print(col)

# Create a DataFrame with column names
feature_df = pd.DataFrame(column_names, columns=['feature'])

# Save the column names as a CSV file
output_path = os.path.join(data_dir, 'feature_list.csv')
feature_df.to_csv(output_path, index=False)

print(f"\nFeature list saved to: {output_path}")

Number of columns: 16223
Column names:
HALLMARK_ADIPOGENESIS
HALLMARK_ALLOGRAFT_REJECTION
HALLMARK_ANDROGEN_RESPONSE
HALLMARK_ANGIOGENESIS
HALLMARK_APICAL_JUNCTION
HALLMARK_APICAL_SURFACE
HALLMARK_APOPTOSIS
HALLMARK_BILE_ACID_METABOLISM
HALLMARK_CHOLESTEROL_HOMEOSTASIS
HALLMARK_COAGULATION
HALLMARK_COMPLEMENT
HALLMARK_DNA_REPAIR
HALLMARK_E2F_TARGETS
HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION
HALLMARK_ESTROGEN_RESPONSE_EARLY
HALLMARK_ESTROGEN_RESPONSE_LATE
HALLMARK_FATTY_ACID_METABOLISM
HALLMARK_G2M_CHECKPOINT
HALLMARK_GLYCOLYSIS
HALLMARK_HEDGEHOG_SIGNALING
HALLMARK_HEME_METABOLISM
HALLMARK_HYPOXIA
HALLMARK_IL2_STAT5_SIGNALING
HALLMARK_IL6_JAK_STAT3_SIGNALING
HALLMARK_INFLAMMATORY_RESPONSE
HALLMARK_INTERFERON_ALPHA_RESPONSE
HALLMARK_INTERFERON_GAMMA_RESPONSE
HALLMARK_KRAS_SIGNALING_DN
HALLMARK_KRAS_SIGNALING_UP
HALLMARK_MITOTIC_SPINDLE
HALLMARK_MTORC1_SIGNALING
HALLMARK_MYC_TARGETS_V1
HALLMARK_MYC_TARGETS_V2
HALLMARK_MYOGENESIS
HALLMARK_NOTCH_SIGNALING
HALLMARK_OXIDATIVE_PHOSPHORYLATION

In [9]:
features_ms = [
    "BUSSLINGER_GASTRIC_IMMUNE_CELLS",
    "FAN_OVARY_CL8_MATURE_CUMULUS_GRANULOSA_CELL_2",
    "MANNO_MIDBRAIN_NEUROTYPES_HENDO",
    "MANNO_MIDBRAIN_NEUROTYPES_HGABA",
    "MANNO_MIDBRAIN_NEUROTYPES_HPERIC",
    "MURARO_PANCREAS_DUCTAL_CELL",
    "TRAVAGLINI_LUNG_PROLIFERATING_MACROPHAGE_CELL",
    "TRAVAGLINI_LUNG_PROXIMAL_CILIATED_CELL",
    "GOBP_ANATOMICAL_STRUCTURE_FORMATION_INVOLVED_IN_MORPHOGENESIS",
    "GOBP_BIOLOGICAL_ADHESION",
    "GOBP_CATION_TRANSPORT",
    "GOBP_CELLULAR_MACROMOLECULE_LOCALIZATION",
    "GOBP_CELLULAR_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_CELLULAR_RESPONSE_TO_STRESS",
    "GOBP_CELL_CELL_SIGNALING",
    "GOBP_CELL_MIGRATION",
    "GOBP_CELL_PROJECTION_ORGANIZATION",
    "GOBP_CENTRAL_NERVOUS_SYSTEM_DEVELOPMENT",
    "GOBP_CHEMICAL_HOMEOSTASIS",
    "GOBP_CIRCULATORY_SYSTEM_DEVELOPMENT",
    "GOBP_CYTOSKELETON_ORGANIZATION",
    "GOBP_ESTABLISHMENT_OF_PROTEIN_LOCALIZATION",
    "GOBP_HOMEOSTATIC_PROCESS",
    "GOBP_INTRACELLULAR_TRANSPORT",
    "GOBP_ION_TRANSMEMBRANE_TRANSPORT",
    "GOBP_ION_TRANSPORT",
    "GOBP_LOCOMOTION",
    "GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_STIMULUS",
    "GOBP_NEGATIVE_REGULATION_OF_SIGNALING",
    "GOBP_NERVOUS_SYSTEM_PROCESS",
    "GOBP_NEUROGENESIS",
    "GOBP_NEURON_DEVELOPMENT",
    "GOBP_NEURON_DIFFERENTIATION",
    "GOBP_NITROGEN_COMPOUND_TRANSPORT",
    "GOBP_ORGANONITROGEN_COMPOUND_BIOSYNTHETIC_PROCESS",
    "GOBP_PHOSPHORYLATION",
    "GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_POSITIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_PROTEIN_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_SIGNALING",
    "GOBP_PROGRAMMED_CELL_DEATH",
    "GOBP_PROTEIN_CONTAINING_COMPLEX_ORGANIZATION",
    "GOBP_PROTEOLYSIS",
    "GOBP_REGULATION_OF_CELL_DEATH",
    "GOBP_REGULATION_OF_CELL_DIFFERENTIATION",
    "GOBP_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION",
    "GOBP_REGULATION_OF_MULTICELLULAR_ORGANISMAL_DEVELOPMENT",
    "GOBP_REGULATION_OF_PHOSPHORUS_METABOLIC_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS",
    "GOBP_REGULATION_OF_TRANSPORT",
    "GOBP_REPRODUCTION",
    "GOBP_RESPONSE_TO_ABIOTIC_STIMULUS",
    "GOBP_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_SMALL_MOLECULE_METABOLIC_PROCESS",
    "GOBP_TISSUE_DEVELOPMENT",
    "GOBP_TRANSMEMBRANE_TRANSPORT",
    "GOBP_VESICLE_MEDIATED_TRANSPORT",
    "GOCC_ENDOPLASMIC_RETICULUM",
    "GOCC_ENVELOPE",
    "GOCC_GOLGI_APPARATUS",
    "GOCC_INTRINSIC_COMPONENT_OF_PLASMA_MEMBRANE",
    "GOCC_MEMBRANE_PROTEIN_COMPLEX",
    "GOCC_MITOCHONDRION",
    "GOCC_NEURON_PROJECTION",
    "GOCC_PLASMA_MEMBRANE_REGION",
    "GOCC_SECRETORY_VESICLE",
    "GOCC_SYNAPSE",
    "GOCC_VESICLE_MEMBRANE",
    "GOMF_IDENTICAL_PROTEIN_BINDING",
    "GOMF_MOLECULAR_FUNCTION_REGULATOR",
    "GOMF_PROTEIN_CONTAINING_COMPLEX_BINDING",
    "GOMF_RIBONUCLEOTIDE_BINDING",
    "GOMF_SIGNALING_RECEPTOR_BINDING",
    "GOMF_TRANSPORTER_ACTIVITY"
]

In [10]:
features_pancreas = [
    "DESCARTES_FETAL_CEREBELLUM_VASCULAR_ENDOTHELIAL_CELLS",
    "GAO_LARGE_INTESTINE_ADULT_CJ_IMMUNE_CELLS",
    "HAY_BONE_MARROW_STROMAL",
    "MANNO_MIDBRAIN_NEUROTYPES_HENDO",
    "MANNO_MIDBRAIN_NEUROTYPES_HMGL",
    "MANNO_MIDBRAIN_NEUROTYPES_HPERIC",
    "MURARO_PANCREAS_ACINAR_CELL",
    "MURARO_PANCREAS_DUCTAL_CELL",
    "MURARO_PANCREAS_MESENCHYMAL_STROMAL_CELL",
    "TRAVAGLINI_LUNG_PROLIFERATING_MACROPHAGE_CELL",
    "GOBP_ANATOMICAL_STRUCTURE_FORMATION_INVOLVED_IN_MORPHOGENESIS",
    "GOBP_ANIMAL_ORGAN_MORPHOGENESIS",
    "GOBP_BIOLOGICAL_ADHESION",
    "GOBP_BIOLOGICAL_PROCESS_INVOLVED_IN_INTERSPECIES_INTERACTION_BETWEEN_ORGANISMS",
    "GOBP_CELLULAR_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_CELLULAR_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_CELLULAR_RESPONSE_TO_STRESS",
    "GOBP_CELL_ACTIVATION",
    "GOBP_CELL_CELL_ADHESION",
    "GOBP_CELL_CELL_SIGNALING",
    "GOBP_CELL_CYCLE",
    "GOBP_CELL_MIGRATION",
    "GOBP_CELL_PROJECTION_ORGANIZATION",
    "GOBP_CHEMICAL_HOMEOSTASIS",
    "GOBP_CIRCULATORY_SYSTEM_DEVELOPMENT",
    "GOBP_CYTOSKELETON_ORGANIZATION",
    "GOBP_DEFENSE_RESPONSE",
    "GOBP_ENZYME_LINKED_RECEPTOR_PROTEIN_SIGNALING_PATHWAY",
    "GOBP_EPITHELIUM_DEVELOPMENT",
    "GOBP_HOMEOSTATIC_PROCESS",
    "GOBP_IMMUNE_RESPONSE",
    "GOBP_INFLAMMATORY_RESPONSE",
    "GOBP_ION_TRANSPORT",
    "GOBP_LIPID_METABOLIC_PROCESS",
    "GOBP_LOCOMOTION",
    "GOBP_NEGATIVE_REGULATION_OF_BIOSYNTHETIC_PROCESS",
    "GOBP_NEGATIVE_REGULATION_OF_CELL_DEATH",
    "GOBP_NEGATIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_NEGATIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_STIMULUS",
    "GOBP_NEGATIVE_REGULATION_OF_SIGNALING",
    "GOBP_NEUROGENESIS",
    "GOBP_NEURON_DIFFERENTIATION",
    "GOBP_NITROGEN_COMPOUND_TRANSPORT",
    "GOBP_PHOSPHORYLATION",
    "GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_POSITIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_GENE_EXPRESSION",
    "GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION",
    "GOBP_POSITIVE_REGULATION_OF_MULTICELLULAR_ORGANISMAL_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_NUCLEOBASE_CONTAINING_COMPOUND_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_PROTEIN_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_RNA_METABOLIC_PROCESS",
    "GOBP_POSITIVE_REGULATION_OF_SIGNALING",
    "GOBP_PROGRAMMED_CELL_DEATH",
    "GOBP_PROTEOLYSIS",
    "GOBP_REGULATION_OF_ANATOMICAL_STRUCTURE_MORPHOGENESIS",
    "GOBP_REGULATION_OF_CELLULAR_COMPONENT_MOVEMENT",
    "GOBP_REGULATION_OF_CELL_ADHESION",
    "GOBP_REGULATION_OF_CELL_DEATH",
    "GOBP_REGULATION_OF_CELL_DIFFERENTIATION",
    "GOBP_REGULATION_OF_CELL_POPULATION_PROLIFERATION",
    "GOBP_REGULATION_OF_IMMUNE_SYSTEM_PROCESS",
    "GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION",
    "GOBP_REGULATION_OF_MULTICELLULAR_ORGANISMAL_DEVELOPMENT",
    "GOBP_REGULATION_OF_PHOSPHORUS_METABOLIC_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS",
    "GOBP_REGULATION_OF_PROTEIN_PHOSPHORYLATION",
    "GOBP_REGULATION_OF_RESPONSE_TO_EXTERNAL_STIMULUS",
    "GOBP_REGULATION_OF_RESPONSE_TO_STRESS",
    "GOBP_REGULATION_OF_TRANSPORT",
    "GOBP_REPRODUCTION",
    "GOBP_RESPONSE_TO_ABIOTIC_STIMULUS",
    "GOBP_RESPONSE_TO_CYTOKINE",
    "GOBP_RESPONSE_TO_ENDOGENOUS_STIMULUS",
    "GOBP_RESPONSE_TO_LIPID",
    "GOBP_RESPONSE_TO_OXYGEN_CONTAINING_COMPOUND",
    "GOBP_SMALL_MOLECULE_METABOLIC_PROCESS",
    "GOBP_TISSUE_DEVELOPMENT",
    "GOBP_TRANSMEMBRANE_TRANSPORT",
    "GOBP_TUBE_DEVELOPMENT",
    "GOBP_TUBE_MORPHOGENESIS",
    "GOBP_VASCULATURE_DEVELOPMENT",
    "GOBP_VESICLE_MEDIATED_TRANSPORT",
    "GOCC_CELL_SURFACE",
    "GOCC_ENDOPLASMIC_RETICULUM",
    "GOCC_GOLGI_APPARATUS",
    "GOCC_INTRINSIC_COMPONENT_OF_PLASMA_MEMBRANE",
    "GOCC_PLASMA_MEMBRANE_REGION",
    "GOMF_IDENTICAL_PROTEIN_BINDING",
    "GOMF_MOLECULAR_FUNCTION_REGULATOR",
    "GOMF_MOLECULAR_TRANSDUCER_ACTIVITY",
    "GOMF_PROTEIN_CONTAINING_COMPLEX_BINDING",
    "GOMF_SEQUENCE_SPECIFIC_DNA_BINDING",
    "GOMF_SIGNALING_RECEPTOR_BINDING",
    "GOMF_TRANSCRIPTION_REGULATOR_ACTIVITY",
    "HP_ABNORMAL_RESPIRATORY_SYSTEM_MORPHOLOGY"
]

In [11]:
# columns_to_consider = ['gene_sequence'] + list(cells_features.columns[103:])
if dataset == 'ms':
    columns_to_consider = ['gene_sequence'] + features_ms
else:
    columns_to_consider = ['gene_sequence'] + features_pancreas
    
print(len(columns_to_consider))

99


In [12]:
# Filtering the dataframe to only include these columns
filtered_cells = cells_features[columns_to_consider]

In [13]:
print('Shape:', filtered_cells.shape)
filtered_cells.head()

Shape: (499, 99)


,gene_sequence,DESCARTES_FETAL_CEREBELLUM_VASCULAR_ENDOTHELIAL_CELLS,GAO_LARGE_INTESTINE_ADULT_CJ_IMMUNE_CELLS,HAY_BONE_MARROW_STROMAL,MANNO_MIDBRAIN_NEUROTYPES_HENDO,MANNO_MIDBRAIN_NEUROTYPES_HMGL,MANNO_MIDBRAIN_NEUROTYPES_HPERIC,MURARO_PANCREAS_ACINAR_CELL,MURARO_PANCREAS_DUCTAL_CELL,MURARO_PANCREAS_MESENCHYMAL_STROMAL_CELL,...,GOCC_INTRINSIC_COMPONENT_OF_PLASMA_MEMBRANE,GOCC_PLASMA_MEMBRANE_REGION,GOMF_IDENTICAL_PROTEIN_BINDING,GOMF_MOLECULAR_FUNCTION_REGULATOR,GOMF_MOLECULAR_TRANSDUCER_ACTIVITY,GOMF_PROTEIN_CONTAINING_COMPLEX_BINDING,GOMF_SEQUENCE_SPECIFIC_DNA_BINDING,GOMF_SIGNALING_RECEPTOR_BINDING,GOMF_TRANSCRIPTION_REGULATOR_ACTIVITY,HP_ABNORMAL_RESPIRATORY_SYSTEM_MORPHOLOGY
0,"<cls>,TXLNA,ATP2A3,ZNF141,CCDC141,SLC25A37,STO...","0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,...","0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,...","0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,...","0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,...","0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,1,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,...",...,"0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...","0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,1,0,0,1,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,..."
1,"<cls>,DSG2,TMEM144,TNFRSF1B,FKBP11,MUC20,DCN,A...","0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,...","0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...","0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,0,0,...","0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,...","0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,..."
2,"<cls>,CNN2,LRAT,ACTN1,PLD6,PHLDB2,PIGA,CES2,MY...","0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,...","0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,...","0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,...","0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,...","0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,..."
3,"<cls>,NINJ1,TNS3,SLC6A6,PGM2L1,CABYR,SLC3A1,SE...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,0,1,0,0,0,

In [14]:
# Export compiled_cells_all-features.csv
num_col = filtered_cells.shape[1]
filtered_cells_path = os.path.join(data_dir, f'filt_features_{num_col}.csv')
filtered_cells.to_csv(filtered_cells_path, index=False, sep=';')

In [15]:
# Extract the column names directly
old_col = list(filtered_cells.columns)
# Create a new DataFrame with these column names as row values in a single column
rename_df = pd.DataFrame(old_col, columns=['old_col'])
# Display the first few rows of the new DataFrame
rename_df.head()

,old_col
0,gene_sequence
1,DESCARTES_FETAL_CEREBELLUM_VASCULAR_ENDOTHELIA...
2,GAO_LARGE_INTESTINE_ADULT_CJ_IMMUNE_CELLS
3,HAY_BONE_MARROW_STROMAL
4,MANNO_MIDBRAIN_NEUROTYPES_HENDO


In [16]:
import os
# Open new column names

col_df_path = os.path.join(data_dir, f'column_names_final.csv')
col_df = pd.read_csv(col_df_path)
col_df.head(10)

,old_col,new_col
0,gene_sequence,gene_sequence
1,DESCARTES_FETAL_CEREBELLUM_VASCULAR_ENDOTHELIA...,fetal cerebellum vascular endothelial cells
2,GAO_LARGE_INTESTINE_ADULT_CJ_IMMUNE_CELLS,large intestine adult immune cells
3,HAY_BONE_MARROW_STROMAL,bone marrow stromal
4,MANNO_MIDBRAIN_NEUROTYPES_HENDO,midbrain neurotype human endothelial cell
5,MANNO_MIDBRAIN_NEUROTYPES_HMGL,midbrain neurotypes human microglial cell
6,MANNO_MIDBRAIN_NEUROTYPES_HPERIC,midbrain neurotype human pericyte
7,MURARO_PANCREAS_ACINAR_CELL,pancreas acinar cell
8,MURARO_PANCREAS_DUCTAL_CELL,pancreas ductal cell
9,MURARO_PANCREAS_MESENCHYMAL_STROMAL_CELL,pancreas mesenchymal stromal cell


In [17]:
filtered_cells.rename(columns=dict(zip(col_df['old_col'], col_df['new_col'])), inplace=True)
filtered_cells.head()

/var/folders/y_/n7t8whz54_q5s7xfy2s01ypw0000gn/T/ipykernel_36898/2923648683.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_cells.rename(columns=dict(zip(col_df['old_col'], col_df['new_col'])), inplace=True)


,gene_sequence,fetal cerebellum vascular endothelial cells,large intestine adult immune cells,bone marrow stromal,midbrain neurotype human endothelial cell,midbrain neurotypes human microglial cell,midbrain neurotype human pericyte,pancreas acinar cell,pancreas ductal cell,pancreas mesenchymal stromal cell,...,GOCC intrinsic component of plasma membrane,GOCC plasma membrane region,GOMF identical protein binding,GOMF molecular function regulator,GOMF molecular transducer activity,GOMF protein containing complex binding,GOMF sequence specific DNA binding,GOMF signaling receptor binding,GOMF transcription regulator activity,abnormal respiratory system morphology
0,"<cls>,TXLNA,ATP2A3,ZNF141,CCDC141,SLC25A37,STO...","0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,...","0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,...","0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,...","0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,...","0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,1,1,1,1,0,0,1,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,...",...,"0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...","0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,1,0,0,1,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,..."
1,"<cls>,DSG2,TMEM144,TNFRSF1B,FKBP11,MUC20,DCN,A...","0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,...","0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...","0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,1,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,0,0,0,...","0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,...","0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,..."
2,"<cls>,CNN2,LRAT,ACTN1,PLD6,PHLDB2,PIGA,CES2,MY...","0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,...","0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,...","0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,...","0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,...","0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,..."
3,"<cls>,NINJ1,TNS3,SLC6A6,PGM2L1,CABYR,SLC3A1,SE...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...","0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [18]:
# Merge the DataFrames on the 'gene_sequence' column
num_col = len(columns_to_consider)
print('Number of Features:', num_col)
merge_filtered_cells = pd.merge(compiled_cells, filtered_cells, on='gene_sequence', how='inner')
merge_filtered_cells.head()

Number of Features: 99


,gene_sequence,label,expression,layer0_head0,layer0_head1,layer0_head2,layer0_head3,layer0_head4,layer0_head5,layer0_head6,...,GOCC intrinsic component of plasma membrane,GOCC plasma membrane region,GOMF identical protein binding,GOMF molecular function regulator,GOMF molecular transducer activity,GOMF protein containing complex binding,GOMF sequence specific DNA binding,GOMF signaling receptor binding,GOMF transcription regulator activity,abnormal respiratory system morphology
0,"<cls>,TXLNA,ATP2A3,ZNF141,CCDC141,SLC25A37,STO...",5,"0.0,48.0,33.0,25.0,5.0,3.0,10.0,28.0,5.0,29.0,...","0.0023745133,0.0032477628,0.003589545,0.003814...","0.0041318787,0.0053796424,0.0041540335,0.00306...","0.0056217974,0.0031721308,0.005039241,0.004776...","0.0034448185,0.0032406966,0.0040769065,0.00487...","0.0038653233,0.0041617695,0.0020131315,0.00385...","0.003914756,0.0034802882,0.005446018,0.0034722...","0.0042973505,0.0035008143,0.0028890239,0.00347...",...,"0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,...","0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...","0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,1,0,0,1,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,..."
1,"<cls>,DSG2,TMEM144,TNFRSF1B,FKBP11,MUC20,DCN,A...",5,"0.0,39.0,7.0,19.0,6.0,17.0,7.0,19.0,26.0,6.0,4...","0.0022430576,0.0024487586,0.003219843,0.003661...","0.004013164,0.0031627351,0.0036624381,0.003953...","0.0042283423,0.0031852438,0.004393687,0.002575...","0.0034300121,0.0023620245,0.0043808324,0.00346...","0.0032802566,0.0028360444,0.0044557275,0.00289...","0.004048269,0.0047665904,0.0038825262,0.003081...","0.0042852582,0.0031577605,0.0056061805,0.00359...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,...","0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,..."
2,"<cls>,CNN2,LRAT,ACTN1,PLD6,PHLDB2,PIGA,CES2,MY...",5,"0.0,27.0,20.0,39.0,5.0,7.0,3.0,11.0,12.0,1.0,3...","0.0022238698,0.0040896675,0.004484682,0.002118...","0.0039243894,0.0040566986,0.0026453421,0.00478...","0.0048712324,0.0034570356,0.0030926508,0.00369...","0.003476348,0.0037290629,0.0028929042,0.004210...","0.003399026,0.0025909895,0.0042975056,0.004100...","0.004021946,0.002352441,0.0031674122,0.0027053...","0.0031837898,0.0027910753,0.0021618963,0.00399...",...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,...","0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,..."
3,"<cls>,NINJ1,TNS3,SLC6A6,PGM2L1,CABYR,SLC3A1,SE...",5,"0.0,42.0,19.0,39.0,20.0,17.0,14.0,33.0,21.0,37...","0.0020445194,0.0031070835,0.0025515305,0.00327...","0.0040934836,0.0028073455,0.0034256505,0.00244...","0.0043800664,0.0029222732,0.0025062195,0.00437...","0.0034533436,0.0025593678,0.003978598,0.002954...","0.0038189043,0.0032943436,0.002777985,0.003936...","0.0036355853,0.0033676857,0.00409,0.0038118216...","0.0043353783,0.0040053073,0.002787665,0.002944...",...,"0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,...","

In [19]:
merge_filtered_cells.shape

(499, 201)

In [20]:
print(data_dir)
print(model)
print(num_col)

/Users/vivianschu/Documents/MBP/Lab/Genomic Interpretability/scgpt_attn_results/test/FINAL_LRP-attention/pancreas/random_init-2
random_init-2
99


In [21]:
final_df_path = os.path.join(data_dir, f'scgpt_{model}_{dataset}-features_{num_col}.csv')
print(final_df_path)
merge_filtered_cells.to_csv(final_df_path, index=False, sep=';')

/Users/vivianschu/Documents/MBP/Lab/Genomic Interpretability/scgpt_attn_results/test/FINAL_LRP-attention/pancreas/random_init-2/scgpt_random_init-2_pancreas-features_99.csv
